# RAG for Predictive Maintenance (LIME Incoporation)

### Table of Contents (Pipeline)

| Section 
|---------
| **Part 0 — Imports & Config** 
| **Part 1 — Load and Dataset Preparation** 
| **Part 2 — Train/Test Split** 
| **Part 3 — Baseline Interpretable Decision Tree** 
| **Part 4 — Random Forest: Teacher Model** 
| **Part 5 — Set Up LIME Tabular Explainer** 
| **Part 6 — Collect LIME Explanations for Failure Cases** 
| **Part 7 — Build LIME-Derived Thresholds** 
| **Part 8 — LIME Threshold Rule-Based Classifier** 
| **Part 9 — Evaluations: Is LIME Thresholds Method Better than the Decision Tree** 

## 0. Imports & Config

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from lime.lime_tabular import LimeTabularExplainer

In [2]:
pip install lime

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 1. Load & Prepare the ai4i2020 Dataset

In [3]:
# Adjust path as needed
df = pd.read_csv("Data_file/ai4i2020.csv")

# Typical ai4i2020 columns:
# 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]',
# 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
# 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'

# ---- Target (binary failure label) ----
# If your label column is named differently, change this:
label_col = "Machine failure"
y = df[label_col].astype(int)

# ---- Feature set ----
# We drop the individual failure-mode columns; we want only the 6 core features
drop_cols = [label_col, "TWF", "HDF", "PWF", "OSF", "RNF"]
X = df.drop(columns=drop_cols)

# Identify categorical and numeric columns
categorical_cols = ["Type"]  # Product ID is unique-ish and not useful to model; we can drop it
if "Product ID" in X.columns:
    X = X.drop(columns=["Product ID"])

numeric_cols = [c for c in X.columns if c not in categorical_cols]

X.head()

,UDI,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,1,M,298.1,308.6,1551,42.8,0
1,2,L,298.2,308.7,1408,46.3,3
2,3,L,298.1,308.5,1498,49.4,5
3,4,L,298.2,308.6,1433,39.5,7
4,5,L,298.2,308.7,1408,40.0,9


## 2. Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y 
)
# 0.3 instead of the normal 0.2 because of the kind of data we're working with.

# Preprocessing: One-Hot for categorical + StandardScaler for numerics
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

## 3. Baseline Interpretable Decision Tree

In [5]:
# Small tree to ensure interpretability (like the base paper)
dt_baseline = DecisionTreeClassifier(
    max_depth=3,        # shallow tree => simple thresholds
    min_samples_leaf=50,
    random_state=42,
    class_weight="balanced"  # handle imbalance a bit
)

dt_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", dt_baseline)
    ]
)

dt_pipeline.fit(X_train, y_train)

print("=== Baseline Decision Tree Performance ===")
y_pred_dt = dt_pipeline.predict(X_test)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, digits=3))

=== Baseline Decision Tree Performance ===
Confusion matrix:
 [[2654  244]
 [  22   80]]
              precision    recall  f1-score   support

           0      0.992     0.916     0.952      2898
           1      0.247     0.784     0.376       102

    accuracy                          0.911      3000
   macro avg      0.619     0.850     0.664      3000
weighted avg      0.966     0.911     0.933      3000



## 4. Random Forest: Teacher Model

In [6]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight="balanced"
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", rf)
    ]
)

rf_pipeline.fit(X_train, y_train)

print("=== Random Forest Performance (Teacher) ===")
y_pred_rf = rf_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_rf, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))


=== Random Forest Performance (Teacher) ===
              precision    recall  f1-score   support

           0      0.984     0.998     0.991      2898
           1      0.900     0.529     0.667       102

    accuracy                          0.982      3000
   macro avg      0.942     0.764     0.829      3000
weighted avg      0.981     0.982     0.980      3000

Confusion matrix:
 [[2892    6]
 [  48   54]]


## 5. Set Up LIME Tabular Explainer

In [7]:
# We need the *transformed* training data and feature names for LIME
rf_preprocessor = rf_pipeline.named_steps["preprocess"]
rf_clf = rf_pipeline.named_steps["clf"]

X_train_transformed = rf_preprocessor.transform(X_train)

# Build feature names after preprocessing
ohe = rf_preprocessor.named_transformers_["cat"]
cat_feature_names = list(ohe.get_feature_names_out(categorical_cols))
all_feature_names = numeric_cols + cat_feature_names

explainer = LimeTabularExplainer(
    training_data=np.array(X_train_transformed),
    feature_names=all_feature_names,
    class_names=["normal", "failure"],
    discretize_continuous=True,
    mode="classification"
)

## 6. Collect LIME Explanations for Failure Cases

In [8]:
def get_rf_proba(X_tab):
    """Wrapper for RF predict_proba with preprocessing."""
    X_trans = rf_preprocessor.transform(X_tab)
    return rf_clf.predict_proba(X_trans)

# We'll only look at a subset of failure cases to keep runtime sane
failure_idx = X_test.index[y_pred_rf == 1]  # RF-predicted failures
sampled_failure_idx = failure_idx[:200]     # at most 200 for explanation

lime_records = []  # store (feature, value, weight, instance_index)

for idx in sampled_failure_idx:
    x_row = X.loc[idx:idx]    # df with single row
    x_trans = rf_preprocessor.transform(x_row)
    exp = explainer.explain_instance(
        data_row=x_trans[0],
        predict_fn=lambda z: rf_clf.predict_proba(z),
        num_features=6
    )
    # LIME returns feature names in transformed space; we want original numeric ones
    for fname, weight in exp.as_list():
        # Skip one-hot encoded categoricals for threshold derivation
        # only care about numeric features (torque, wear, speed, temps)
        for orig in numeric_cols:
            if orig in fname:
                lime_records.append(
                    {
                        "instance": idx,
                        "feature": orig,
                        "weight": weight,
                        "value": float(x_row[orig].values[0])
                    }
                )
                break

lime_df = pd.DataFrame(lime_records)
lime_df.head()

,instance,feature,weight,value
0,6078,Torque [Nm],0.045444,59.1
1,6078,Tool wear [min],0.038558,203.0
2,6078,Rotational speed [rpm],0.022696,1339.0
3,6078,Process temperature [K],0.005657,310.7
4,6078,UDI,-0.005592,6079.0


## 7. Build LIME-Derived Thresholds

In [9]:
# Simple domain assumptions: high torque, high wear, high process temp,
# low rotational speed are risky. Air temperature effect is weaker but we include it.
direction = {
    "Torque [Nm]": "high",
    "Tool wear [min]": "high",
    "Process temperature [K]": "high",
    "Air temperature [K]": "high",
    "Rotational speed [rpm]": "low"
}

thresholds = {}

for feat in numeric_cols:
    feat_lime = lime_df[(lime_df["feature"] == feat) & (lime_df["weight"] > 0)]
    if feat_lime.empty:
        continue

    vals = feat_lime["value"].values

    if direction.get(feat, "high") == "high":
        thr = np.percentile(vals, 70)  # high-risk region above this
    else:
        thr = np.percentile(vals, 30)  # low-risk region below this

    thresholds[feat] = {
        "direction": direction.get(feat, "high"),
        "threshold": thr,
        "n_samples_used": len(vals)
    }

thresholds

{'UDI': {'direction': 'high',
  'threshold': np.float64(4596.8),
  'n_samples_used': 35},
 'Air temperature [K]': {'direction': 'high',
  'threshold': np.float64(303.0),
  'n_samples_used': 37},
 'Process temperature [K]': {'direction': 'high',
  'threshold': np.float64(310.7),
  'n_samples_used': 11},
 'Rotational speed [rpm]': {'direction': 'low',
  'threshold': np.float64(1313.5),
  'n_samples_used': 52},
 'Torque [Nm]': {'direction': 'high',
  'threshold': np.float64(59.190000000000005),
  'n_samples_used': 48},
 'Tool wear [min]': {'direction': 'high',
  'threshold': np.float64(209.4),
  'n_samples_used': 23}}

## 8. LIME Threshold Rule-Based Classifier

In [10]:
def lime_rule_predict(X_df, thresholds, min_risk_features=1):
    """
    X_df: original feature DataFrame (with original column names)
    thresholds: dict from above
    min_risk_features: how many violated thresholds before calling it failure
    """
    risk_scores = []

    for _, row in X_df.iterrows():
        risk = 0
        for feat, info in thresholds.items():
            if feat not in row.index:
                continue
            val = row[feat]
            if info["direction"] == "high":
                if val > info["threshold"]:
                    risk += 1
            else:  # 'low'
                if val < info["threshold"]:
                    risk += 1
        risk_scores.append(risk)

    risk_scores = np.array(risk_scores)
    y_pred = (risk_scores >= min_risk_features).astype(int)
    return y_pred, risk_scores

# Predictions on test set
y_pred_lime, risk_scores_test = lime_rule_predict(X_test, thresholds, min_risk_features=1)

print("=== LIME Threshold Rule-Based Model Performance ===")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lime))
print(classification_report(y_test, y_pred_lime, digits=3))

=== LIME Threshold Rule-Based Model Performance ===
Confusion matrix:
 [[1044 1854]
 [  19   83]]
              precision    recall  f1-score   support

           0      0.982     0.360     0.527      2898
           1      0.043     0.814     0.081       102

    accuracy                          0.376      3000
   macro avg      0.512     0.587     0.304      3000
weighted avg      0.950     0.376     0.512      3000



## 9. Evaluations

## 9.1 Compare Against Decision Tree & RF (Fidelity)

In [11]:
# (1) Ground truth comparison is already printed via classification_report.

# (2) Fidelity to Random Forest
def agreement(a, b):
    return np.mean(a == b)

agree_dt_rf = agreement(y_pred_dt, y_pred_rf)
agree_lime_rf = agreement(y_pred_lime, y_pred_rf)

print(f"Agreement (Decision Tree vs Random Forest): {agree_dt_rf:.3f}")
print(f"Agreement (LIME Rule vs Random Forest):     {agree_lime_rf:.3f}")

Agreement (Decision Tree vs Random Forest): 0.906
Agreement (LIME Rule vs Random Forest):     0.364


## 9.2 Recall on Failure

In [12]:
from sklearn.metrics import recall_score

print("Recall on failure (Decision Tree):", recall_score(y_test, y_pred_dt))
print("Recall on failure (LIME-rule):", recall_score(y_test, y_pred_lime))

Recall on failure (Decision Tree): 0.7843137254901961
Recall on failure (LIME-rule): 0.8137254901960784


## Due to the higher recall score, LIME model is catching more failures

## 9.3 Compute Precision on Failure

In [13]:
from sklearn.metrics import precision_score

print("\n=== FAILURE CLASS PRECISION ===")
print("Decision Tree:", precision_score(y_test, y_pred_dt))
print("LIME Threshold:", precision_score(y_test, y_pred_lime))


=== FAILURE CLASS PRECISION ===
Decision Tree: 0.24691358024691357
LIME Threshold: 0.04284976768198245


## LIME may have lower precision — because it marks “failure” more aggressively. That’s expected and good for safety.

## 9.4 Compute F1 Score for Failure

In [14]:
from sklearn.metrics import f1_score

# This balances percision + recall

print("\n=== FAILURE CLASS F1 ===")
print("Decision Tree:", f1_score(y_test, y_pred_dt))
print("LIME Threshold:", f1_score(y_test, y_pred_lime))


=== FAILURE CLASS F1 ===
Decision Tree: 0.3755868544600939
LIME Threshold: 0.08141245708680726


## 9.5 False Negative Rate

In [15]:
import numpy as np

def false_negative_rate(y_true, y_pred):
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    return fn / (fn + tp)

print("\n=== FALSE NEGATIVE RATE ===")
print("Decision Tree:", false_negative_rate(y_test, y_pred_dt))
print("LIME Threshold:", false_negative_rate(y_test, y_pred_lime))


=== FALSE NEGATIVE RATE ===
Decision Tree: 0.21568627450980393
LIME Threshold: 0.18627450980392157


## False negatives = “actual failure but prediction = normal” (very bad in industrial safety)
## Lower is better. LIME has lower FNR → that’s a major contribution.

## 9.6 Confusion Matrices (Failure Emphasis)

In [16]:
print("\n=== CONFUSION MATRIX — Decision Tree ===")
print(confusion_matrix(y_test, y_pred_dt))

print("\n=== CONFUSION MATRIX — LIME Threshold ===")
print(confusion_matrix(y_test, y_pred_lime))


=== CONFUSION MATRIX — Decision Tree ===
[[2654  244]
 [  22   80]]

=== CONFUSION MATRIX — LIME Threshold ===
[[1044 1854]
 [  19   83]]


## 9.7 SAFETY INDEX (introducing a NEW research metric)

## A model with high recall and acceptable FP is good.

In [17]:
from sklearn.metrics import confusion_matrix

def safety_index(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn)
    fpr = fp / (fp + tn)
    return recall / (1 + fpr) 

print("\n=== SAFETY INDEX ===")
print("Decision Tree:", safety_index(y_test, y_pred_dt))
print("LIME Threshold:", safety_index(y_test, y_pred_lime))


=== SAFETY INDEX ===
Decision Tree: 0.723405848653911
LIME Threshold: 0.49624925727866903
